# Lab 5 — Build an OpenAI-Compatible API
**Day 2 Morning | ~45 minutes | Colab CPU**

---

## What You Will Build
By the end of this lab you will have:
1. Written a FastAPI server that exposes `/v1/chat/completions` (OpenAI wire format)
2. Launched it with a public tunnel URL anyone can call
3. Called your own endpoint with the OpenAI Python client — indistinguishable from the real API
4. Added streaming SSE responses
5. Compared three providers with identical client code (your server, OpenAI direct, Groq)

> **The key idea:** You are *being* the backend today, not calling one.
> Every vLLM, TGI, and Ollama server exposes exactly this shape. Now you know why.

In [ ]:
%%capture
!pip install fastapi uvicorn pyngrok openai httpx
print('Done')

In [ ]:
# Configuration — works in Colab Secrets or local environment variables
import os

def get_secret(name: str, *, required: bool = True):
    value = os.environ.get(name)
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass
    if required and not value:
        raise ValueError(
            f"Missing {name}. In Colab, open the key icon in the left sidebar, "
            f"add a secret named {name}, paste the value, and enable Notebook access."
        )
    return value

OPENAI_API_KEY   = get_secret('OPENAI_API_KEY')
NGROK_AUTH_TOKEN = get_secret('NGROK_AUTH_TOKEN')
OPENAI_BASE_URL  = 'https://api.openai.com/v1'
DEFAULT_MODEL    = 'gpt-4o-mini'

print(f'Config loaded — OpenAI key starts with {OPENAI_API_KEY[:8]}..., ngrok token found')


### How to Read the Server Cell

The next code cell writes a complete `server.py` file. It is dense because a web server needs models, request schemas, routes, and streaming behavior in one file. Read it in four passes:

1. **Configuration:** environment variables choose the upstream backend.
2. **Schemas:** Pydantic models define the request shape.
3. **Routes:** `/health`, `/v1/models`, and `/v1/chat/completions` mimic the OpenAI API.
4. **Streaming:** Server-Sent Events send token deltas as they arrive.

The deployment lesson is not FastAPI syntax. The lesson is that your application can talk to this server, OpenAI, vLLM, LiteLLM, or another compatible backend with the same client shape.


---

## Part A — Write the Server (20 min)

We write `server.py` from a notebook cell, then launch it as a background process.
The server proxies incoming requests to OpenAI — your client code never changes.

In [ ]:
# Cell A1 — Write server.py
# INSTRUCTOR NOTE: 'Read this top to bottom. Four endpoints. One file. This is a real serving pattern.'
server_code = '''
import os, json, time, uuid
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import List, Optional
from openai import OpenAI

app = FastAPI(title="My LLM API", version="0.1.0",
              description="OpenAI-compatible server backed by OpenAI (swap backend = change env vars)")

BACKEND_API_KEY  = os.environ.get("BACKEND_API_KEY", "")
BACKEND_BASE_URL = os.environ.get("BACKEND_BASE_URL", "https://api.openai.com/v1")
DEFAULT_MODEL    = os.environ.get("DEFAULT_MODEL",    "gpt-4o-mini")

backend = OpenAI(api_key=BACKEND_API_KEY, base_url=BACKEND_BASE_URL)

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = DEFAULT_MODEL
    messages: List[Message]
    stream: bool = False
    temperature: float = 0.7
    max_tokens: Optional[int] = 500

@app.get("/health")
def health():
    return {"status": "ok", "backend": BACKEND_BASE_URL, "model": DEFAULT_MODEL}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [
        {"id": DEFAULT_MODEL, "object": "model", "owned_by": "custom-server"}
    ]}

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    msgs = [{"role": m.role, "content": m.content} for m in req.messages]

    if not req.stream:
        resp = backend.chat.completions.create(
            model=req.model, messages=msgs,
            temperature=req.temperature, max_tokens=req.max_tokens
        )
        return {
            "id":      f"chatcmpl-{uuid.uuid4().hex[:8]}",
            "object":  "chat.completion",
            "created": int(time.time()),
            "model":   req.model,
            "choices": [{"index": 0,
                         "message": {"role": "assistant",
                                     "content": resp.choices[0].message.content},
                         "finish_reason": "stop"}],
            "usage":   {"prompt_tokens":     resp.usage.prompt_tokens,
                        "completion_tokens": resp.usage.completion_tokens,
                        "total_tokens":      resp.usage.total_tokens}
        }

    def generate():
        stream = backend.chat.completions.create(
            model=req.model, messages=msgs,
            temperature=req.temperature, stream=True
        )
        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                data = {"id":      f"chatcmpl-{uuid.uuid4().hex[:8]}",
                        "object":  "chat.completion.chunk",
                        "created": int(time.time()),
                        "model":   req.model,
                        "choices": [{"index": 0,
                                     "delta": {"content": delta},
                                     "finish_reason": None}]}
                yield f"data: {json.dumps(data)}\\n\\n"
        yield "data: [DONE]\\n\\n"

    return StreamingResponse(generate(), media_type="text/event-stream")
'''

with open('server.py', 'w') as f:
    f.write(server_code)
print('✅ server.py written')

In [ ]:
# Cell A2 — Launch server + open public tunnel
import subprocess, time, os
from pyngrok import ngrok, conf

# Set ngrok auth token (required — sign up free at ngrok.com)
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Pass our OpenAI key to the server via environment
env = {**os.environ, 'BACKEND_API_KEY': OPENAI_API_KEY,
       'BACKEND_BASE_URL': OPENAI_BASE_URL, 'DEFAULT_MODEL': DEFAULT_MODEL}

server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server:app', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=env
)
time.sleep(3)

public_url = ngrok.connect(8000)
SERVER_URL  = str(public_url)

print(f'✅ Server running!')
print(f'   Public URL : {SERVER_URL}')
print(f'   Swagger UI : {SERVER_URL}/docs   ← open this in your browser!')
print(f'   Health     : {SERVER_URL}/health')


---

## Part B — Call Your Server (15 min)

Your server is live. Now prove it speaks the OpenAI protocol.

In [ ]:
# Cell B1 — Health check
import httpx
r = httpx.get(f'{SERVER_URL}/health')
print('Health:', r.json())

In [ ]:
# Cell B2 — OpenAI client calling YOUR server
# INSTRUCTOR NOTE: 'This is the payoff. The client does not know which server it is talking to.'
from openai import OpenAI

my_client = OpenAI(base_url=f'{SERVER_URL}/v1', api_key='not-needed')

resp = my_client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[
        {'role': 'system', 'content': 'You are an LLM deployment expert.'},
        {'role': 'user',   'content': 'What are 3 advantages of vLLM over a naive FastAPI server?'}
    ]
)
print(resp.choices[0].message.content)
print(f'\nTokens: {resp.usage}')

In [ ]:
# Cell B3 — Streaming from your server
print('Streaming from our server:')
print('─' * 60)
stream = my_client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{'role': 'user', 'content': 'List 5 things that can go wrong when serving LLMs. Be brief.'}],
    stream=True
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print('\n' + '─' * 60)

In [ ]:
# Cell B4 — Provider comparison: three backends, identical client code
# INSTRUCTOR NOTE: 'Change only base_url + api_key. The rest is identical.'
providers = {
    'Our FastAPI → OpenAI': {
        'base_url': f'{SERVER_URL}/v1', 'api_key': 'not-needed', 'model': DEFAULT_MODEL
    },
    'OpenAI direct (4o-mini)': {
        'base_url': OPENAI_BASE_URL, 'api_key': OPENAI_API_KEY, 'model': 'gpt-4o-mini'
    },
    'OpenAI direct (4o)': {
        'base_url': OPENAI_BASE_URL, 'api_key': OPENAI_API_KEY, 'model': 'gpt-4o'
    },
}

question = 'In one sentence: what is PagedAttention?'
print(f'Question: {question}\n')
for name, cfg in providers.items():
    c = OpenAI(base_url=cfg['base_url'], api_key=cfg['api_key'])
    r = c.chat.completions.create(
        model=cfg['model'], messages=[{'role': 'user', 'content': question}]
    )
    print(f'[{name}]')
    print(f'  {r.choices[0].message.content}\n')

In [ ]:
# Cell B5 — What vLLM adds on top of what we built
print('''
OUR FASTAPI SERVER (built today)     vs     vLLM
─────────────────────────────────────────────────────────────────
✅ OpenAI-compatible /v1/chat/completions   ✅ Same
✅ Streaming SSE                             ✅ Same
✅ Easy to read and modify                   ❌ More complex
❌ Sequential: one request at a time         ✅ Continuous batching: N concurrent users
❌ No KV-cache management                   ✅ PagedAttention: non-contiguous KV pages
❌ No GPU memory optimisation               ✅ 20–30x throughput vs naive serving
❌ No quantisation support built-in         ✅ AWQ / GPTQ / bitsandbytes native

When to use ours: development, testing, internal tools with low traffic
When to use vLLM: production, real user traffic, GPU cost-sensitive deployments

Production vLLM command (run on a GPU server, not Colab):
  python -m vllm.entrypoints.openai.api_server \\
    --model Qwen/Qwen2.5-7B-Instruct \\
    --quantization awq \\
    --host 0.0.0.0 --port 8000
  → Drop-in replacement for our server. Same client code.
''')

In [ ]:
# Cleanup
server_proc.terminate()
ngrok.disconnect(public_url)
print('Server stopped.')

---

## ✅ Lab 5 Complete

You should now have:
- [ ] `server.py` written and launched
- [ ] Public Swagger UI URL opened in a browser (or phone)
- [ ] OpenAI client successfully calling your own endpoint
- [ ] Streaming printed token by token
- [ ] Three-provider comparison printed from identical client code

## Stretch Goals

1. **Request logging:** Add a `@app.middleware('http')` that logs timestamp, prompt character count, and latency for every request.
2. **API key auth:** Add a check in `chat_completions` that returns HTTP 401 if an `Authorization: Bearer` header is missing or wrong.
3. **Metrics endpoint:** Add `GET /metrics` that returns total request count, average latency, and last error message.
4. **Swap the backend to Groq:** Change `BACKEND_BASE_URL` and `BACKEND_API_KEY` env vars to use Groq.
   All client code stays the same. This is the LiteLLM pattern.
5. **LiteLLM in one command:** Run `pip install litellm` then `litellm --model openai/gpt-4o-mini`.
   It creates a ready-made proxy in one CLI command.